# Interactive Generation — ConvVAE + CEM

- **Mode 1** — `z_seed` picks a fixed latent vector; digit sliders intervene on the two concepts while z stays the same.
- **Mode 2** — every slider move samples a **brand-new** z from N(0,1) and decodes it with the chosen digit concepts.

In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

from scripts.train_cem_vae import load_cem_vae

CHECKPOINT    = "../outputs/models/cem_vae_mnist"
CONCEPT_MEAN  = 5.0
CONCEPT_SCALE = 4.0

device = "cpu"
if torch.backends.mps.is_available():  device = "mps"
elif torch.cuda.is_available():         device = "cuda"

model, backbone_cfg, cem_cfg = load_cem_vae(CHECKPOINT, device=device)
print(f"Loaded on {device} | n_concepts={cem_cfg.n_concepts}, latent_dim={backbone_cfg.latent_dim}")

Loaded on mps | n_concepts=2, latent_dim=32


In [2]:
def generate(z, digit1, digit2):
    c = torch.tensor(
        [[(digit1 - CONCEPT_MEAN) / CONCEPT_SCALE,
          (digit2 - CONCEPT_MEAN) / CONCEPT_SCALE]],
        device=device
    )
    with torch.no_grad():
        emb, _ = model.cem(z, interventions=c)
        recon  = model.decoder(emb).reconstruction
    img = recon[0].squeeze().cpu().float().numpy()
    return (img * 0.3081 + 0.1307).clip(0, 1)

## Mode 1 — Fixed z, move the concept sliders

Change **z seed** to pick a different latent vector from the prior.  
Then move **Digit 1** and **Digit 2** to intervene on the concepts while the same z is kept.

In [ ]:
@interact(
    z_seed = IntSlider(min=0, max=99, value=0,  description='z seed:',  continuous_update=False),
    digit1 = IntSlider(min=1, max=9,  value=5,  description='Digit 1:', continuous_update=False),
    digit2 = IntSlider(min=1, max=9,  value=5,  description='Digit 2:', continuous_update=False),
)
def mode1(z_seed=0, digit1=5, digit2=5):
    torch.manual_seed(z_seed)
    z   = torch.randn(1, backbone_cfg.latent_dim, device=device)
    img = generate(z, digit1, digit2)
    plt.figure(figsize=(3, 3))
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.title(f"seed={z_seed}  |  digit1={digit1}, digit2={digit2}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## Mode 2 — New z on every slider move

Every time you change a slider, a **fresh z** is sampled from N(0,1) and decoded with the digit values you set.

In [ ]:
@interact(
    digit1 = IntSlider(min=1, max=9, value=5, description='Digit 1:', continuous_update=False),
    digit2 = IntSlider(min=1, max=9, value=5, description='Digit 2:', continuous_update=False),
)
def mode2(digit1=5, digit2=5):
    z   = torch.randn(1, backbone_cfg.latent_dim, device=device)
    img = generate(z, digit1, digit2)
    plt.figure(figsize=(3, 3))
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.title(f"fresh z  |  digit1={digit1}, digit2={digit2}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()